# K-fold cross validation

Implement a random k-fold cross validation algorithm from scratch.

Your algorithm should:
- load the iris dataset and split its columns into features and target
- split the dataset into k-fold to perform cross validation

You can use the code bellow to implement your algorithm or implement yourself from scratch.



In [3]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris

# load the iris dataset
iris = load_iris()
X = iris.data
y = iris.target


In [4]:
print(X.shape)
print(y.shape)

(150, 4)
(150,)


In [5]:
print(iris.feature_names)

['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']


In [6]:
print(iris.target_names)

['setosa' 'versicolor' 'virginica']


In [7]:
print(y)

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2]


In [8]:
# we will implement a k-fold cross validation from scratch
# we will use the iris dataset

def k_fold_cross_validation(X, y, k, model, random_state=42, verbose=True):
    # X is the data
    # y is the target
    # k is the number of folds
    # model is the model to use
    # we will return the mean accuracy of the model

    #################################
    # 1) Shuffle the data (random k-fold) and split it into k folds
    #    so that X_folds[0] returns the first fold of the data, same for y_folds
    #################################
    X = np.asarray(X)
    y = np.asarray(y)
    n_samples = X.shape[0]

    rng = np.random.default_rng(random_state)
    indices = rng.permutation(n_samples)          # random order of the rows
    X_shuffled, y_shuffled = X[indices], y[indices]

    # np.array_split handles the case where n_samples is not divisible by k
    X_folds = np.array_split(X_shuffled, k)
    y_folds = np.array_split(y_shuffled, k)

    #################################
    # 2) Loop over the folds: each fold is used exactly once as the test set,
    #    the k-1 remaining folds form the training set
    #################################
    accuracies = []
    for i in range(k):
        X_test, y_test = X_folds[i], y_folds[i]
        X_train = np.concatenate([X_folds[j] for j in range(k) if j != i])
        y_train = np.concatenate([y_folds[j] for j in range(k) if j != i])

        # we fit the model on the train data
        model.fit(X_train, y_train)

        # we predict on the test data
        y_pred = model.predict(X_test)

        # we compute the accuracy
        accuracy = np.mean(y_pred == y_test)
        accuracies.append(accuracy)

        if verbose:
            print(f"Fold {i + 1}/{k} - train: {len(X_train)} rows, test: {len(X_test)} rows, accuracy: {accuracy:.4f}")

    # we return the mean accuracy
    if verbose:
        print(f"\nMean accuracy: {np.mean(accuracies):.4f} (+/- {np.std(accuracies):.4f})")
    return np.mean(accuracies)

In [9]:
#You can use the code below to test your function

#import the random forest model
from sklearn.ensemble import RandomForestClassifier

# we will use the random forest model
model = RandomForestClassifier(random_state=42)

# we will use the k_fold_cross_validation function
k_fold_cross_validation(X, y, 5, model)

Fold 1/5 - train: 120 rows, test: 30 rows, accuracy: 0.9333
Fold 2/5 - train: 120 rows, test: 30 rows, accuracy: 0.9667
Fold 3/5 - train: 120 rows, test: 30 rows, accuracy: 0.9667
Fold 4/5 - train: 120 rows, test: 30 rows, accuracy: 0.9333
Fold 5/5 - train: 120 rows, test: 30 rows, accuracy: 0.9667

Mean accuracy: 0.9533 (+/- 0.0163)


np.float64(0.9533333333333334)

Fold 5/5 - train: 120 rows, test: 30 rows, accuracy: 0.9667

Mean accuracy: 0.9533 (+/- 0.0163)


np.float64(0.9533333333333334)

### Why shuffle?
The iris target is sorted (50 × setosa, then 50 × versicolor, then 50 × virginica). Without shuffling, a fold could contain only one or two classes, so the model would be tested on a class it barely saw during training. Shuffling (random k-fold) avoids that.

### Sanity check
We compare our implementation with scikit-learn's `KFold` + `cross_val_score` (used **only** to check our result, not in our implementation).

In [10]:
from sklearn.model_selection import KFold, cross_val_score

sk_scores = cross_val_score(RandomForestClassifier(random_state=42), X, y,
                            cv=KFold(n_splits=5, shuffle=True, random_state=42))
print("scikit-learn mean accuracy:", round(sk_scores.mean(), 4))

# effect of k on our implementation
for k in [3, 5, 10]:
    acc = k_fold_cross_validation(X, y, k, RandomForestClassifier(random_state=42), verbose=False)
    print(f"k = {k:2d} -> mean accuracy = {acc:.4f}")

scikit-learn mean accuracy: 0.96
k =  3 -> mean accuracy = 0.9533
k =  5 -> mean accuracy = 0.9533
k = 10 -> mean accuracy = 0.9533


k =  3 -> mean accuracy = 0.9533


k =  5 -> mean accuracy = 0.9533


k = 10 -> mean accuracy = 0.9533


**Conclusion:** our from-scratch k-fold gives a mean accuracy of about 95 % on iris with a Random Forest, very close to scikit-learn's result. Small differences come from the random shuffling (different folds), not from the method itself.